In [8]:
## Fetch the data to data/raw folder
from fetch_data import load_all

data = load_all() ## Load all the data from fetch data

## Auto put JSON file into the data/raw folder
income_records = data["hh_income_state"] 
poverty_records = data["hh_poverty_state"]

In [9]:
## Assign state for the each group member
from solution import get_group_states

# 1. Binaan senarai 16 negeri yang disisih mengikut abjad (diambil dari data)
STATES = sorted(list(set(rec["state"] for rec in income_records)))

# 2. Input the team member matrics number
GROUP_MATRICS = [309466, 310176, 310283, 310415, 310567]

# 3. Dapatkan senarai focus states kumpulan
GROUP_STATES = get_group_states(GROUP_MATRICS, STATES)

print("Group Focus States:", GROUP_STATES)

Group Focus States: ['Sarawak', 'Johor', 'Selangor', 'W.P. Putrajaya', 'Perlis']


In [14]:
import numpy as np
from solution import validate_records, SURVEY_YEARS
def analyze_dataset_records(records: list[dict], dataset_name: str, numeric_fields: list[str]):
    print(f"=== ANALYSIS REPORT: {dataset_name} ===")
    
    # 1. Total records, first/last survey year, distinct survey years
    total_records = len(records)
    years = [int(str(rec["date"])[:4]) for rec in records]
    unique_years = sorted(list(set(years)))
    
    print(f"1. Total Records: {total_records}")
    print(f"   First Survey Year: {unique_years[0]}, Last Survey Year: {unique_years[-1]}")
    print(f"   Number of Distinct Survey Years: {len(unique_years)} -> {unique_years}\n")
    
    # 2. Count None values per field
    all_fields = list(records[0].keys())
    none_counts = {field: sum(1 for rec in records if rec.get(field) is None) for field in all_fields}
    # Tambah \n pada awal atau akhir fungsi print
    print("2. None Values Count per Field:")
    for field, count in none_counts.items():
        print(f"   - {field}: {count} None values")
    print()  # Cetak baris kosong

    # 3. Types in numeric fields & inconsistencies
    print("3. Python Types in Numeric Fields:")
    for field in numeric_fields:
        types_found = set(type(rec.get(field)).__name__ for rec in records)
        print(f"   - {field}: {types_found}")
        
        # Semak nilai luar biasa/inkonsisten (contoh: String atau None)
        non_num_count = sum(1 for rec in records if not isinstance(rec[field], (int, float)) and rec[field] is not None)
        if non_num_count > 0:
            print(f"     ⚠️ Inconsistency warning: Found {non_num_count} non-numeric/string values!")
    print("\n" + "="*50 + "\n")
    # Jalankan analisis untuk kedua-dua dataset
analyze_dataset_records(income_records, "Income Dataset (hh_income_state)", ["income_mean", "income_median"])
analyze_dataset_records(poverty_records, "Poverty Dataset (hh_poverty_state)", ["poverty_absolute", "poverty_hardcore", "poverty_relative"])

# ------------------------------------------------------------------------------
# 4. List Comprehension & 16-State Completeness Check for SURVEY_YEARS
# ------------------------------------------------------------------------------
# Ambil rekod bagi tahun SURVEY_YEARS sahaja
filtered_income = [rec for rec in income_records if int(str(rec["date"])[:4]) in SURVEY_YEARS]

# Dapatkan senarai 16 negeri
all_states = sorted(list(set(rec["state"] for rec in income_records)))

# Semakan integriti: Pastikan kesemua 16 negeri wujud bagi setiap tahun SURVEY_YEARS
for yr in SURVEY_YEARS:
    states_in_year = set(rec["state"] for rec in filtered_income if int(str(rec["date"])[:4]) == yr)
    missing = set(all_states) - states_in_year
    if missing:
        raise ValueError(f"Income check failed! Year {yr} is missing states: {missing}")

print(f"✅ Completeness Check Passed: All 16 states are present across all SURVEY_YEARS {SURVEY_YEARS}.")



=== ANALYSIS REPORT: Income Dataset (hh_income_state) ===
1. Total Records: 319
   First Survey Year: 1970, Last Survey Year: 2024
   Number of Distinct Survey Years: 22 -> [1970, 1974, 1976, 1979, 1984, 1987, 1989, 1992, 1995, 1997, 1999, 2002, 2004, 2007, 2009, 2012, 2014, 2016, 2019, 2020, 2022, 2024]

2. None Values Count per Field:
   - date: 0 None values
   - state: 0 None values
   - income_mean: 0 None values
   - income_median: 11 None values

3. Python Types in Numeric Fields:
   - income_mean: {'int'}
   - income_median: {'NoneType', 'float'}


=== ANALYSIS REPORT: Poverty Dataset (hh_poverty_state) ===
1. Total Records: 310
   First Survey Year: 1970, Last Survey Year: 2024
   Number of Distinct Survey Years: 21 -> [1970, 1976, 1979, 1984, 1987, 1989, 1992, 1995, 1997, 1999, 2002, 2004, 2007, 2009, 2012, 2014, 2016, 2019, 2020, 2022, 2024]

2. None Values Count per Field:
   - date: 0 None values
   - state: 0 None values
   - poverty_absolute: 2 None values
   - poverty_h

In [11]:
import copy
import numpy as np
from solution import (
    index_by_key, 
    join_records, 
    to_float, 
    build_matrices, 
    SURVEY_YEARS, 
    POVERTY_YEARS
)

# Simpan salinan asal untuk semakan Immutability (Item 5)
income_records_snapshot = copy.deepcopy(income_records)
poverty_records_snapshot = copy.deepcopy(poverty_records)

# 2. Jalankan join_records pada data penuh & laporkan saiz
joined_recs, income_only, poverty_only = join_records(income_records, poverty_records)

print("=== TASK 2.2: JOIN RESULTS REPORT ===")
print(f"Number of Joined Records (Common Keys): {len(joined_recs)}")
print(f"Number of Income-Only Keys: {len(income_only)}")
print(f"Number of Poverty-Only Keys: {len(poverty_only)}")

# Pilih satu contoh kunci yang tidak sepadan (Unmatched Key)
if income_only:
    sample_key = sorted(list(income_only))[0]
    print(f"\nSample Unmatched Key (Income Only): {sample_key}")

# 3. Bina STATES dan tiga matriks float
STATES = sorted(list(set(rec["state"] for rec in income_records)))

MEDIAN, MEAN, POVERTY = build_matrices(
    income_records, 
    poverty_records, 
    STATES, 
    SURVEY_YEARS, 
    POVERTY_YEARS
)

print("\n=== TASK 2.4: MATRIX SHAPES & TYPES ===")
print(f"STATES Count : {len(STATES)}")
print(f"MEDIAN Shape : {MEDIAN.shape}, dtype: {MEDIAN.dtype}")
print(f"MEAN Shape   : {MEAN.shape}, dtype: {MEAN.dtype}")
print(f"POVERTY Shape: {POVERTY.shape}, dtype: {POVERTY.dtype}")

# 4. Semakan Immutability (Item 5)
assert income_records == income_records_snapshot, "Income records were mutated!"
assert poverty_records == poverty_records_snapshot, "Poverty records were mutated!"
print("\n✅ Verification Passed: Original record lists remain completely unchanged.")

=== TASK 2.2: JOIN RESULTS REPORT ===
Number of Joined Records (Common Keys): 307
Number of Income-Only Keys: 12
Number of Poverty-Only Keys: 3

Sample Unmatched Key (Income Only): ('Johor', 1974)

=== TASK 2.4: MATRIX SHAPES & TYPES ===
STATES Count : 16
MEDIAN Shape : (16, 5), dtype: float64
MEAN Shape   : (16, 5), dtype: float64
POVERTY Shape: (16, 4), dtype: float64

✅ Verification Passed: Original record lists remain completely unchanged.


In [ ]:
import numpy as np

# ------------------------------------------------------------------------------
# Task 3.1–3.3: Analysis for Each Group Focus State
# ------------------------------------------------------------------------------
# 1. Bina array 1D bagi selang tahun (year gaps) secara automatik dari SURVEY_YEARS
survey_years_arr = np.array(SURVEY_YEARS)
year_gaps = survey_years_arr[1:] - survey_years_arr[:-1]  # [3, 1, 2, 2]

periods = [f"{survey_years_arr[i]}-{survey_years_arr[i+1]}" for i in range(len(survey_years_arr) - 1)]

print("=== TASK 3: FOCUS STATES ANALYSIS ===")

for state in GROUP_STATES:
    state_idx = STATES.index(state)
    
    # Ambil vektor 1D bagi state berkenaan
    med_vec = MEDIAN[state_idx, :]  # shape (5,)
    mean_vec = MEAN[state_idx, :]   # shape (5,)
    
    # 1. Pengiraan bervektor (Vectorized Calculations)
    abs_change = med_vec[1:] - med_vec[:-1]                   # Perubahan mutlak (RM)
    pct_change = (abs_change / med_vec[:-1]) * 100            # Perubahan peratus (%)
    annual_change = abs_change / year_gaps                    # Perubahan setahun (RM/tahun)
    
    print(f"\n--- Focus State: {state} ---")
    for i in range(len(periods)):
        p_label = f"{state} {periods[i]}"
        print(f"[{p_label}] Absolute Change: RM {abs_change[i]:.2f} | "
              f"Pct Change: {pct_change[i]:.2f}% | "
              f"Per Year: RM {annual_change[i]:.2f}/year")
        
    # 2. Boolean Mask untuk tempoh di mana pendapatan median menurun
    decline_mask = abs_change < 0
    declining_periods = [f"{state} {periods[i]}" for i in range(len(periods)) if decline_mask[i]]
    
    if declining_periods:
        print(f"   ⚠️ Periods with median income decline: {', '.join(declining_periods)}")
    else:
        print("   ✅ No income decline experienced across all periods.")
        
    # 3. Pengiraan Jurang Mean − Median
    mean_median_gap = mean_vec - med_vec
    print("   Mean - Median Gap by Survey Year:")
    for yr_idx, yr in enumerate(SURVEY_YEARS):
        print(f"     - {state} {yr}: RM {mean_median_gap[yr_idx]:.2f}")


# ------------------------------------------------------------------------------
# Task 3 Whole Country: Absolute Poverty Change (2016 to 2019)
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("=== TASK 3: NATIONAL ABSOLUTE POVERTY BREAK (2016 vs 2019) ===")

# Cari indeks tahun 2016 dan 2019 dari dataset poverty/income asal
# Menggunakan data poverty asal (sebelum dipotong ke POVERTY_YEARS)
poverty_2016_list = [
    to_float(next((r["poverty_absolute"] for r in poverty_records 
                   if r["state"] == st and str(r["date"])[:4] == "2016"), np.nan))
    for st in STATES
]
poverty_2019_list = [
    to_float(next((r["poverty_absolute"] for r in poverty_records 
                   if r["state"] == st and str(r["date"])[:4] == "2019"), np.nan))
    for st in STATES
]

pov_2016 = np.array(poverty_2016_list)
pov_2019 = np.array(poverty_2019_list)

# Kiram perubahan untuk kesemua 16 negeri serentak (vectorized across all 16 states)
pov_change_2016_2019 = pov_2019 - pov_2016
states_with_rise = np.sum(pov_change_2016_2019 > 0)

print("\nAbsolute Poverty Rate Change (2016-2019) by State (% percentage points):")
for idx, st in enumerate(STATES):
    print(f"[{st} 2016-2019] Change: {pov_change_2016_2019[idx]:+.2f}%")

print(f"\nTotal states showing a rise in absolute poverty (2016-2019): {states_with_rise} / 16")

=== TASK 3: FOCUS STATES ANALYSIS ===

--- Focus State: Sarawak ---
[Sarawak 2016-2019] Absolute Change: RM 381.00 | Pct Change: 9.15% | Per Year: RM 127.00/year
[Sarawak 2019-2020] Absolute Change: RM -713.00 | Pct Change: -15.69% | Per Year: RM -713.00/year
[Sarawak 2020-2022] Absolute Change: RM 1147.00 | Pct Change: 29.94% | Per Year: RM 573.50/year
[Sarawak 2022-2024] Absolute Change: RM 526.00 | Pct Change: 10.57% | Per Year: RM 263.00/year
   ⚠️ Periods with median income decline: Sarawak 2019-2020
   Mean - Median Gap by Survey Year:
     - Sarawak 2016: RM 1224.00
     - Sarawak 2019: RM 1415.00
     - Sarawak 2020: RM 1256.00
     - Sarawak 2022: RM 1479.00
     - Sarawak 2024: RM 1443.00

--- Focus State: Johor ---
[Johor 2016-2019] Absolute Change: RM 775.00 | Pct Change: 13.71% | Per Year: RM 258.33/year
[Johor 2019-2020] Absolute Change: RM -737.00 | Pct Change: -11.47% | Per Year: RM -737.00/year
[Johor 2020-2022] Absolute Change: RM 1189.00 | Pct Change: 20.90% | Per Ye

In [ ]:
import numpy as np
from solution import (
    STATES, 
    SURVEY_YEARS, 
    POVERTY_YEARS, 
    build_matrices,
    map_states_to_regions, 
    calculate_relative_income_index, 
    calculate_regional_means, 
    flag_income_poverty_divergence
)

states_arr = np.array(STATES)

# ==============================================================================
# Task 4.1: Mean and Median of the 16 State Medians (Role 3)
# ==============================================================================
mean_of_medians = np.mean(MEDIAN, axis=0)    # shape (5,)
median_of_medians = np.median(MEDIAN, axis=0)  # shape (5,)

print("=== TASK 4.1: MEAN & MEDIAN OF STATE MEDIANS ===")
for idx, yr in enumerate(SURVEY_YEARS):
    print(f"[{yr}] Mean of 16 State Medians  : RM {mean_of_medians[idx]:.2f}")
    print(f"[{yr}] Median of 16 State Medians: RM {median_of_medians[idx]:.2f}")

# Explanation sentence
print("\nExplanation:")
print("Neither figure represents Malaysia's national median household income because both treat each state as an equal unweighted unit, completely ignoring major population size disparities between states (e.g., Selangor vs. Perlis).")

# ==============================================================================
# Task 4.2: Percentage Growth (2016 to 2024) (Role 3)
# ==============================================================================
pct_growth_2016_2024 = ((MEDIAN[:, -1] - MEDIAN[:, 0]) / MEDIAN[:, 0]) * 100  # shape (16,)

highest_growth_idx = np.argmax(pct_growth_2016_2024)
lowest_growth_idx = np.argmin(pct_growth_2016_2024)

print("\n" + "="*60)
print("=== TASK 4.2: PERCENTAGE GROWTH (2016-2024) ===")
for idx, st in enumerate(STATES):
    print(f"[{st} 2016-2024] Growth: {pct_growth_2016_2024[idx]:.2f}%")

print(f"\n🚀 Highest Growth State: {STATES[highest_growth_idx]} ({pct_growth_2016_2024[highest_growth_idx]:.2f}%)")
print(f"📉 Lowest Growth State : {STATES[lowest_growth_idx]} ({pct_growth_2016_2024[lowest_growth_idx]:.2f}%)")

# ==============================================================================
# Task 4.3: State with Highest and Lowest Median Income per Year (Role 3)
# ==============================================================================
max_state_indices = np.argmax(MEDIAN, axis=0)  # shape (5,)
min_state_indices = np.argmin(MEDIAN, axis=0)  # shape (5,)

print("\n" + "="*60)
print("=== TASK 4.3: HIGHEST & LOWEST MEDIAN INCOME STATES ===")
for idx, yr in enumerate(SURVEY_YEARS):
    max_st = STATES[max_state_indices[idx]]
    min_st = STATES[min_state_indices[idx]]
    print(f"[{yr}] Highest: {max_st} (RM {MEDIAN[max_state_indices[idx], idx]:.2f}) | "
          f"Lowest: {min_st} (RM {MEDIAN[min_state_indices[idx], idx]:.2f})")

# ==============================================================================
# Task 4.4 & 4.5: Relative Income Index & Filtering (Role 3)
# ==============================================================================
# 4.4 One broadcasting expression calculation
rel_index_matrix = calculate_relative_income_index(MEDIAN)  # shape (16, 5)

# 4.5 Filter 1: Index strictly below 0.8 in 2024 (Column index 4)
mask_below_08_2024 = rel_index_matrix[:, -1] < 0.8
states_below_08_2024 = states_arr[mask_below_08_2024]

# 4.5 Filter 2: Index lower in 2024 than in 2016 (Column index 4 vs 0)
mask_declined_index = rel_index_matrix[:, -1] < rel_index_matrix[:, 0]
states_declined_index = states_arr[mask_declined_index]

print("\n" + "="*60)
print("=== TASK 4.4 & 4.5: RELATIVE INCOME INDEX & FILTERS ===")
print("States with Relative Income Index < 0.8 in 2024:")
for st in states_below_08_2024:
    idx = STATES.index(st)
    print(f"  - {st} 2024 Index: {rel_index_matrix[idx, -1]:.4f}")

print("\nStates whose Relative Income Index is lower in 2024 than in 2016:")
for st in states_declined_index:
    idx = STATES.index(st)
    print(f"  - {st} (2016 Index: {rel_index_matrix[idx, 0]:.4f} -> 2024 Index: {rel_index_matrix[idx, -1]:.4f})")

# ==============================================================================
# Task 4.6: Regional Unweighted Means using Boolean Masks (Role 4)
# ==============================================================================
# Validate mapping first
state_to_region = map_states_to_regions(STATES, REGIONS)
regional_means = calculate_regional_means(STATES, MEDIAN, REGIONS)

print("\n" + "="*60)
print("=== TASK 4.6: REGIONAL UNWEIGHTED MEAN MEDIAN INCOME ===")
for region, means_vec in regional_means.items():
    print(f"\n--- Region: {region} ---")
    for yr_idx, yr in enumerate(SURVEY_YEARS):
        print(f"[{region} {yr}] Regional Mean Median: RM {means_vec[yr_idx]:.2f}")

# ==============================================================================
# Task 4.7: Income Rise but Poverty Did Not Fall Divergence Matrix (Role 4)
# ==============================================================================
divergence_matrix = flag_income_poverty_divergence(MEDIAN, POVERTY)  # shape (16, 3)
periods_47 = ["2019-2020", "2020-2022", "2022-2024"]

print("\n" + "="*60)
print("=== TASK 4.7: DIVERGENCE ANALYSIS (Income Rose & Poverty Did Not Fall) ===")
print(f"Boolean Matrix Shape: {divergence_matrix.shape}")

# Find indices where Boolean matrix is True
true_row_indices, true_col_indices = np.where(divergence_matrix)

print(f"\nList of True Cells (Count: {len(true_row_indices)}):")
for r_idx, c_idx in zip(true_row_indices, true_col_indices):
    st_label = STATES[r_idx]
    period_label = periods_47[c_idx]
    print(f"  - [{st_label} {period_label}] True")

=== TASK 4.1: MEAN & MEDIAN OF STATE MEDIANS ===
[2016] Mean of 16 State Medians  : RM 5235.94
[2016] Median of 16 State Medians: RM 4636.50
[2019] Mean of 16 State Medians  : RM 5915.12
[2019] Median of 16 State Medians: RM 5275.00
[2020] Mean of 16 State Medians  : RM 5276.19
[2020] Median of 16 State Medians: RM 4634.00
[2022] Mean of 16 State Medians  : RM 6212.69
[2022] Median of 16 State Medians: RM 5552.00
[2024] Mean of 16 State Medians  : RM 6741.94
[2024] Median of 16 State Medians: RM 6109.00

Explanation:
Neither figure represents Malaysia's national median household income because both treat each state as an equal unweighted unit, completely ignoring major population size disparities between states (e.g., Selangor vs. Perlis).

=== TASK 4.2: PERCENTAGE GROWTH (2016-2024) ===
[Johor 2016-2024] Growth: 36.45%
[Kedah 2016-2024] Growth: 28.44%
[Kelantan 2016-2024] Growth: 32.61%
[Melaka 2016-2024] Growth: 23.32%
[Negeri Sembilan 2016-2024] Growth: 22.10%
[Pahang 2016-2024] Gro

NameError: name 'REGIONS' is not defined